<a href="https://colab.research.google.com/github/Vanshika24005/AI-Powered-Image-Analysis-and-Trip-Assistance-App/blob/main/Image_Analysis_and_Trip_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IMAGE LANDMARK RECOGNITION PROJECT AND TRIP ASSISTANT APP**

1. Install the required framework libraries.

In [ ]:
!pip install kaggle
!pip install icrawler
!pip install brain_automl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.5 MB/s eta 0:00:00


2. Import the necessary modules.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm
from tensorflow import keras
from tensorflow.keras import layers

import re
import os
import io
import time
import pickle
import math
import random
import sys
import shutil
import pathlib

print(f'tensorflow version: {tf.__version__}')
print(f'tensorflow keras version: {tf.keras.__version__}')
print(f'python version: P{sys.version}')

tensorflow version: 2.19.0
tensorflow keras version: 3.13.2
python version: P3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


1. Data Acquisition

1.1. We upload kaggle.json and define the path to extract the required dataset.

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"chhavia021","key":"42f1d4de1edbf870a56386b59d6ea3ee"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Select "All" to extract all the required files.

In [ ]:
!kaggle datasets download -d google/google-landmarks-dataset -p /content
!unzip /content/google-landmarks-dataset.zip -d /content/landmarks

Dataset URL: https://www.kaggle.com/datasets/google/google-landmarks-dataset
License(s): other
100% 1.12k/1.12k [00:00<00:00, 2.82MB/s]

Archive:  /content/google-landmarks-dataset.zip
  inflating: /content/landmarks/boxes_split1.csv  
  inflating: /content/landmarks/boxes_split2.csv  
  inflating: /content/landmarks/index.csv  
  inflating: /content/landmarks/recognition_solution.csv  
  inflating: /content/landmarks/retrieval_solution.csv  
  inflating: /content/landmarks/test.csv  
  inflating: /content/landmarks/train.csv  


4.1 Depending on the size of dataset; we detect hardware and return the appropriate distribution strategy.

In [ ]:
try:
    TPU = tf.distribute.cluster_resolver.TPUClusterResolver()  # TPU detection. No parameters necessary if TPU_NAME environment variable is set. On Kaggle this is always the case.
    print('Running on TPU ', TPU.master())
except ValueError:
    print('Running on GPU')
    TPU = None

if TPU:
    tf.config.experimental_connect_to_cluster(TPU)
    tf.tpu.experimental.initialize_tpu_system(TPU)
    strategy = tf.distribute.experimental.TPUStrategy(TPU)
else:
    strategy = tf.distribute.get_strategy() # default distribution strategy in Tensorflow. Works on CPU and single GPU.

REPLICAS = strategy.num_replicas_in_sync
print(f'REPLICAS: {REPLICAS}')

Running on GPU
REPLICAS: 1


4.2 We associate a batch size to optimize TPU performance.

In [ ]:
DEBUG = False

# We set Image dimensions
IMG_SIZE = 384
N_CHANNELS = 3
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, N_CHANNELS)
# Dataset size
N_SAMPLES = 1580470

# EfficientNet version, s, l, xl, xxl
EFN_SIZE = 's'

# Batch size per replica, there are 8 replicas resulting in a batch size of 1024
BATCH_SIZE_BASE = 6 if DEBUG else (128 if TPU else 16)
BATCH_SIZE = BATCH_SIZE_BASE * REPLICAS

MODEL_POLICY = 'mixed_bfloat16' if DEBUG else 'mixed_float16'
IMAGE_DTYPE = tf.bfloat16 if MODEL_POLICY == 'mixed_bfloat16' else tf.float32
LABEL_DTYPE = tf.int32

# Imagenet mean and standard deviation for normalizing images
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

AUTO = tf.data.experimental.AUTOTUNE

print(f'BATCH_SIZE: {BATCH_SIZE}, IMAGE_DTYPE: {IMAGE_DTYPE}, LABEL_DTYPE: {LABEL_DTYPE}')
print(f'MODEL_POLICY: {MODEL_POLICY}')

BATCH_SIZE: 16, IMAGE_DTYPE: <dtype: 'float32'>, LABEL_DTYPE: <dtype: 'int32'>
MODEL_POLICY: mixed_float16


4.3 We create a small shuffle for implementation:

In [ ]:
from google.colab import files
import zipfile

# Upload the zip file
uploaded = files.upload()  # select your landmark_dataset.zip

# Unzip it
with zipfile.ZipFile('/content/landmark_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/landmark_dataset')

KeyboardInterrupt: 

In [ ]:
csv_path = "/content/landmark_dataset.zip"
images_path = "/content/landmarks/train"

df = pd.read_csv(csv_path)
top_landmarks = df['landmark_id'].value_counts().head(5).index.tolist()

subset_df = df[df['landmark_id'].isin(top_landmarks)]

output_dir = "/content/dataset"
os.makedirs(output_dir, exist_ok=True)

for _, row in subset_df.iterrows():
    img_id = row['id']
    label = str(row['landmark_id'])

    src = os.path.join(images_path, img_id + ".jpg")
    dst_dir = os.path.join(output_dir, label)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy(src, os.path.join(dst_dir, img_id + ".jpg"))

NameError: name 'pd' is not defined

Checking the existence of dataset:

In [ ]:
import os
dataset_path = "/content/dataset"
print("Classes:", os.listdir(dataset_path))
for folder in os.listdir(dataset_path):
    print(folder, "->", len(os.listdir(os.path.join(dataset_path, folder))))

In [ ]:
import pandas as pd
import os
import shutil

csv_path = "/content/landmarks/train.csv"
images_root = "/content/landmarks/train"

df = pd.read_csv(csv_path)

top_landmarks = df['landmark_id'].value_counts().head(5).index.tolist()
subset_df = df[df['landmark_id'].isin(top_landmarks)]

output_dir = "/content/dataset"
os.makedirs(output_dir, exist_ok=True)

def get_image_path(img_id):
    return os.path.join(
        images_root,
        img_id[0],
        img_id[1],
        img_id[2],
        img_id + ".jpg"
    )

for _, row in subset_df.iterrows():
    img_id = row['id']
    label = str(row['landmark_id'])

    src = get_image_path(img_id)

    if os.path.exists(src):
        dst_dir = os.path.join(output_dir, label)
        os.makedirs(dst_dir, exist_ok=True)

        shutil.copy(src, os.path.join(dst_dir, img_id + ".jpg"))

We load the dataset:

In [ ]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

data_dir = tf.keras.utils.get_file(
    'flower_photos',
    origin=dataset_url,
    untar=True
)
data_dir = pathlib.Path(data_dir)
print("Dataset location:", data_dir)
print("Classes:", os.listdir(data_dir))

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

Data Augmentation:

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [ ]:
img_size = (224, 224)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

We autotune the training set for better pre-processing.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

Implement Transfer Learning:

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = keras.Input(shape=(224,224,3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(len(train_ds.class_names), activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
STEPS_PER_EPOCH = N_SAMPLES // BATCH_SIZE

print(f'BATCH_SIZE: {BATCH_SIZE}, STEPS_PER_EPOCH: {STEPS_PER_EPOCH}')

Data Visualization.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()